# PRJEB30386 — аннотация человеческих последовательностей с помощью IgBLAST

Используется общая система аннотации с конфигурацией для человека: AIRR-C/OGRDB V/D/J и `outfmt 19`. FASTQ→FASTA и выход AIRR создаются атомарно; полнота проверяется точным последовательным совпадением `sequence_id`, включая `BARCODE`.


In [ ]:
import csv, gzip, hashlib, json, os, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

DATASET = "PRJEB30386"
ORGANISM = "human"
THREADS = min(8, os.cpu_count() or 1)
HEARTBEAT_SECONDS = 30

ENV_CANDIDATES = [Path(os.environ.get("BCR_ENV", "/opt/conda/envs/bcr_env")), Path("/Users/epishkin/mamba/envs/bcr_env")]
BCR_ENV = next((p for p in ENV_CANDIDATES if (p / "bin/igblastn").is_file()), None)
if BCR_ENV is None: raise FileNotFoundError("bcr_env with igblastn was not found")
IGBLASTN = BCR_ENV / "bin/igblastn"
IGDATA = BCR_ENV / "share/igblast"
DB_DIR = IGDATA / "database/airr_c_human"
V_DB = DB_DIR / "airr_c_human_ig.V"
D_DB = DB_DIR / "airr_c_human_igh.D"
J_DB = DB_DIR / "airr_c_human_ig.J"
AUX = IGDATA / "optional_file/human_gl.aux"

repo = Path.cwd().resolve()
while repo != repo.parent and not (repo / "scripts").is_dir(): repo = repo.parent
if (Path("/data/user/epishkin/results") / DATASET / "merged/fastq").is_dir():
    RESULTS_ROOT = Path("/data/user/epishkin/results")
else:
    RESULTS_ROOT = repo / "results"
MERGED_DIR = RESULTS_ROOT / DATASET / "merged/fastq"
OUT_DIR = RESULTS_ROOT / DATASET / "annotation/igblast"
FASTA_DIR = OUT_DIR / "fasta"
LOG_DIR = OUT_DIR / "logs"
for d in (OUT_DIR, FASTA_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

SAMPLES = sorted(p.name.removesuffix("_assemble-pass.fastq.gz") for p in MERGED_DIR.glob("*_assemble-pass.fastq.gz"))
assert len(SAMPLES) == 4, f"Expected 4 merged samples, found {SAMPLES}"
for p in (IGBLASTN, AUX): assert p.is_file(), p
for db in (V_DB, D_DB, J_DB): assert Path(str(db) + ".nhr").is_file(), db
os.environ["IGDATA"] = str(IGDATA)
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ.get("PATH", "")
print({"dataset":DATASET,"samples":SAMPLES,"threads":THREADS,"bcr_env":str(BCR_ENV),"igdata":str(IGDATA),"output":str(OUT_DIR)})


In [ ]:
def fasta_ids(path):
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                yield line[1:].rstrip("\n\r").split(None, 1)[0]

def airr_ids(path):
    with open(path, newline="") as fh:
        reader = csv.DictReader(fh, delimiter="\t")
        if not reader.fieldnames or "sequence_id" not in reader.fieldnames:
            raise ValueError(f"AIRR sequence_id column missing: {path}")
        for row in reader:
            yield row["sequence_id"]

def validate_exact_ids(fasta, airr):
    n = 0
    fi, ai = fasta_ids(fasta), airr_ids(airr)
    while True:
        f = next(fi, None); a = next(ai, None)
        if f is None or a is None:
            if f != a: raise ValueError(f"ID cardinality mismatch after {n:,} rows: FASTA={f!r}, AIRR={a!r}")
            break
        if f != a: raise ValueError(f"ID/order mismatch at row {n+1:,}: FASTA={f!r}, AIRR={a!r}")
        n += 1
    if n == 0: raise ValueError("AIRR output contains no data rows")
    return n

def fastq_to_fasta_atomic(fastq, fasta):
    tmp = fasta.with_suffix(fasta.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    n = 0
    with gzip.open(fastq, "rt") as src, open(tmp, "w") as dst:
        while True:
            h = src.readline()
            if not h: break
            seq, plus, qual = src.readline(), src.readline(), src.readline()
            if not qual or not h.startswith("@") or not plus.startswith("+"):
                raise ValueError(f"Malformed FASTQ near record {n+1}: {fastq}")
            dst.write(">" + h[1:].rstrip("\n\r") + "\n" + seq)
            n += 1
            if n % 500000 == 0: print(f"  convert {fastq.name}: {n:,}", flush=True)
    if n == 0: raise ValueError(f"No FASTQ records: {fastq}")
    tmp.replace(fasta)
    return n

def run_igblast_atomic(fasta, airr, log, expected):
    tmp = airr.with_suffix(airr.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    cmd = [str(IGBLASTN), "-germline_db_V",str(V_DB), "-germline_db_D",str(D_DB), "-germline_db_J",str(J_DB),
           "-organism",ORGANISM, "-ig_seqtype","Ig", "-domain_system","imgt", "-auxiliary_data",str(AUX),
           "-query",str(fasta), "-outfmt","19", "-num_threads",str(THREADS), "-out",str(tmp)]
    t0 = time.time()
    with open(log, "w") as lf:
        proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)
        print(f"  IgBLAST PID={proc.pid}; sample={fasta.stem}; log={log}", flush=True)
        while proc.poll() is None:
            elapsed = (time.time()-t0)/60
            mb = tmp.stat().st_size/1e6 if tmp.exists() else 0
            print(f"  heartbeat sample={fasta.stem} elapsed={elapsed:.1f}min output={mb:.1f}MB PID={proc.pid}", flush=True)
            time.sleep(HEARTBEAT_SECONDS)
    if proc.returncode != 0: raise RuntimeError(f"IgBLAST rc={proc.returncode}; see {log}")
    observed = validate_exact_ids(fasta, tmp)
    if observed != expected: raise ValueError(f"Expected {expected:,}, observed {observed:,}")
    tmp.replace(airr)
    return observed, (time.time()-t0)/60

def annotate_sample(sample):
    fq = MERGED_DIR / f"{sample}_assemble-pass.fastq.gz"
    fa = FASTA_DIR / f"{sample}.fasta"
    airr = OUT_DIR / f"{sample}.airr.tsv"
    log = LOG_DIR / f"{sample}.igblast.log"
    manifest = OUT_DIR / f"{sample}.manifest.json"
    if fa.exists() and airr.exists():
        try:
            n = validate_exact_ids(fa, airr)
            print(f"[{sample}] validated existing: {n:,}", flush=True); return n
        except Exception as e: print(f"[{sample}] stale output: {e}; rebuilding", flush=True)
    print(f"[{sample}] FASTQ -> FASTA", flush=True)
    n = fastq_to_fasta_atomic(fq, fa)
    print(f"[{sample}] IgBLAST: {n:,} sequences", flush=True)
    observed, minutes = run_igblast_atomic(fa, airr, log, n)
    meta = {"dataset":DATASET,"sample":sample,"organism":ORGANISM,"input_fastq":str(fq),"input_records":n,"airr_rows":observed,
            "database":"AIRR-C/OGRDB human, NCBI archive 2024-01-09","v_db":str(V_DB),"d_db":str(D_DB),"j_db":str(J_DB),
            "auxiliary":str(AUX),"igblast_version":"1.22.0","threads":THREADS,"elapsed_min":minutes,"completed_at":datetime.now(timezone.utc).isoformat()}
    manifest.write_text(json.dumps(meta,indent=2)+"\n")
    print(f"[{sample}] DONE rows={observed:,} elapsed={minutes:.1f}min", flush=True)
    return observed
print("Annotation helpers ready")


In [ ]:
# Полный запуск: четыре библиотеки отдельных цепей и одна конфигурация для человека.
summary = {}
for sample in SAMPLES:
    summary[sample] = annotate_sample(sample)
assert sum(summary.values()) == 3_820_274, summary
(OUT_DIR / "annotation_summary.json").write_text(json.dumps({"dataset":DATASET,"total_airr_rows":sum(summary.values()),"samples":summary}, indent=2)+"\n")
print("ANNOTATION_COMPLETE", json.dumps(summary), "total", sum(summary.values()), flush=True)
